# 02 — Judge Scoring

Run all 50 evaluation instances through each of the three Ollama judge models and record
the raw scores.

**Prerequisites:**
- Notebook 01 has been run (answer corpus and human scores exist in `data/`)
- `ollama` is installed (`make setup` handles this)
- Models are pulled: `make setup` pulls `qwen2.5:1.5b`, `qwen2.5:3b`, `gemma3:4b`

> **Do NOT run `ollama serve` manually.** This notebook starts and restarts Ollama
> automatically. A parallel terminal instance causes "address already in use" errors.

**Outputs produced:**
- `data/eval/scores_qwen2_5_1_5b.json`
- `data/eval/scores_qwen2_5_3b.json`
- `data/eval/scores_gemma3_4b.json`

> **RAM constraint:** Models are scored sequentially. `gemma3:4b` requires ~3.8 GB — do not
> run two judges simultaneously in the same Codespace.

In [4]:
import math
import socket
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import httpx
import json
import time

import pandas as pd

from src.config import load_settings
from src.judging.judge import OllamaJudge
from src.judging.runner import METRICS

settings = load_settings(ROOT / "config" / "settings.yaml")
print(f"Ollama URL: {settings.ollama_url}")
print(f"Models: {settings.models}")

FIGURES_DIR = ROOT / "outputs" / "figures"
RESULTS_DIR = ROOT / "outputs" / "results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ── helpers ───────────────────────────────────────────────────────────────────

def _nan_count(records: list) -> int:
    return sum(1 for r in records if isinstance(r.get("score"), float) and math.isnan(r["score"]))


def _port_free(port: int = 11434) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def ensure_swap(size_gb: int = 4) -> None:
    """Create a swap file if none is active.

    Uses dd instead of fallocate — fallocate creates sparse files on the
    overlay/tmpfs filesystems used in Codespaces, which swapon rejects
    with exit 255 ("Invalid argument"). dd writes real zeros, guaranteeing
    a non-sparse file that swapon accepts.
    """
    swap_info = Path("/proc/swaps").read_text()
    if swap_info.count("\n") > 1:
        print(f"Swap already active:\n{swap_info.strip()}")
        return
    swapfile = "/swapfile"
    subprocess.run(["sudo", "rm", "-f", swapfile], capture_output=True)
    print(f"No swap — creating {size_gb} GB swapfile via dd (takes ~30 s)...", end=" ", flush=True)
    r = subprocess.run(
        ["sudo", "dd", "if=/dev/zero", f"of={swapfile}",
         "bs=1M", f"count={size_gb * 1024}", "status=none"],
        capture_output=True,
    )
    if r.returncode != 0:
        print(f"WARNING: dd failed: {r.stderr.decode().strip()}")
        return
    subprocess.run(["sudo", "chmod", "600", swapfile], capture_output=True)
    subprocess.run(["sudo", "mkswap", swapfile], capture_output=True)
    r = subprocess.run(["sudo", "swapon", swapfile], capture_output=True)
    if r.returncode == 0:
        print("done.")
    else:
        print(f"WARNING: swapon failed: {r.stderr.decode().strip()}")


def drop_page_cache() -> None:
    """Release Linux page/slab/dentry caches (the buff/cache shown in free -m).

    On an 8 GB Codespace this typically frees 1-2 GB right before a model loads.
    """
    result = subprocess.run(
        ["sudo", "sh", "-c", "sync; echo 3 > /proc/sys/vm/drop_caches"],
        capture_output=True,
    )
    if result.returncode == 0:
        print("  Page cache dropped.")
    else:
        print(f"  drop_caches skipped (no sudo or not supported): {result.stderr.decode().strip()}")


def ensure_ollama_running(ollama_url: str, timeout: int = 60) -> bool:
    """Start Ollama if it is not already running. Returns True when ready."""
    if not _port_free():
        return True  # already running
    print("Ollama not running — starting ollama serve...", end=" ", flush=True)
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            httpx.get(f"{ollama_url}/api/tags", timeout=3.0)
            print("ready.")
            return True
        except Exception:
            time.sleep(2)
    print("WARNING: Ollama did not start within timeout.")
    return False


def restart_ollama(ollama_url: str, timeout: int = 60) -> None:
    """Kill Ollama, drop page cache, then start a fresh instance."""
    print("  Restarting Ollama...", end=" ", flush=True)
    subprocess.run(["pkill", "-TERM", "-f", "ollama"], capture_output=True)
    for _ in range(20):  # wait up to 10 s for port to be released
        if _port_free():
            break
        time.sleep(0.5)
    else:
        subprocess.run(["pkill", "-KILL", "-f", "ollama"], capture_output=True)
        time.sleep(2)
    # Free kernel page/slab caches now that Ollama has released its memory.
    drop_page_cache()
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            httpx.get(f"{ollama_url}/api/tags", timeout=3.0)
            print("ready.")
            return
        except Exception:
            time.sleep(2)
    print("WARNING: Ollama did not restart within timeout.")

## 1. Load answer corpus

In [5]:
answers_dir = ROOT / "data" / "answers"

all_instances = []
for path in sorted(answers_dir.glob("*.json")):
    data = json.loads(path.read_text())
    if isinstance(data, list):
        all_instances.extend(data)
    else:
        all_instances.append(data)

print(f"Loaded {len(all_instances)} instances from {answers_dir}")
pd.DataFrame(all_instances).groupby(["pipeline", "question_type"]).size().rename("count").reset_index()

Loaded 50 instances from /workspaces/llm_judge_benchmark/data/answers


,pipeline,question_type,count
0,graph,absence_reasoning,5
1,graph,multi_hop,6
2,graph,single_hop,9
3,handcrafted,absence_reasoning,3
4,handcrafted,multi_hop,3
5,handcrafted,single_hop,4
6,vector,absence_reasoning,5
7,vector,multi_hop,6
8,vector,single_hop,9


## 2. Check Ollama availability

In [6]:
ensure_swap()           # create 4 GB swapfile if not present (one-time, ~5 s)
ensure_ollama_running(settings.ollama_url)

try:
    resp = httpx.get(f"{settings.ollama_url}/api/tags", timeout=5.0)
    available_models = [m["name"] for m in resp.json().get("models", [])]
    print(f"Ollama is running. Available models: {available_models}")
except Exception as e:
    print(f"Ollama not reachable: {e}")
    print("Check that 'ollama' is installed: curl -fsSL https://ollama.com/install.sh | sh")
    available_models = []


No swap detected — creating 4 GB swap file... 

CalledProcessError: Command '['sudo', 'swapon', '/swapfile']' returned non-zero exit status 255.

## 3. Score all instances

Each model is scored independently. Results are written to `data/eval/` after each model
so progress survives interruptions. Ollama is restarted before each model for a clean
memory slate — no manual terminal commands needed.

Estimated times on a 2-CPU Codespace (includes Ollama restart overhead):
- `qwen2.5:1.5b`: ~15 min
- `qwen2.5:3b`: ~20 min
- `gemma3:4b`: ~30 min


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# FORCE_RESCORE = True  → always re-score, even if a valid file exists.
# FORCE_RESCORE = False → skip models with complete, NaN-free score files only.
#                         Models with missing files OR any NaN scores are re-scored automatically.
FORCE_RESCORE = False

eval_dir = ROOT / "data" / "eval"
eval_dir.mkdir(parents=True, exist_ok=True)

print("Score file status:\n")
for model in settings.models:
    safe = model.replace(":", "_").replace(".", "_")
    out_path = eval_dir / f"scores_{safe}.json"
    if out_path.exists():
        records = json.loads(out_path.read_text())
        nans = _nan_count(records)
        if not records:
            tag, detail, action = "[EMPTY]", "0 records", "will score"
        elif nans > 0:
            tag = "[NaN]"
            detail = f"{len(records)} records, {nans} NaN"
            action = "will RE-SCORE (NaN found)" if not FORCE_RESCORE else "will RE-SCORE (forced)"
        else:
            tag = "[OK]"
            detail = f"{len(records)} records, 0 NaN"
            action = "SKIP" if not FORCE_RESCORE else "will RE-SCORE (forced)"
    else:
        tag, detail, action = "[MISSING]", "—", "will score"
    print(f"  {tag:<10} {out_path.name:<32} {detail:<26} → {action}")

In [ ]:
def score_model(model: str, instances: list, output_dir: Path) -> list:
    """Score all instances with one judge using a single combined call per instance."""
    judge = OllamaJudge(model=model, ollama_url=settings.ollama_url, num_ctx=settings.num_ctx)
    records = []
    total = len(instances)
    t0 = time.perf_counter()

    for inst_idx, instance in enumerate(instances, 1):
        try:
            scores = judge.score_all_metrics(
                question=str(instance["question"]),
                context=instance["context"],
                answer=str(instance["answer"]),
            )
        except Exception as exc:
            print(f"  ERROR {instance['id']}: {exc}")
            scores = {m: float("nan") for m in METRICS}

        for metric, score in scores.items():
            records.append({"id": instance["id"], "model": model, "metric": metric, "score": score})

        elapsed = time.perf_counter() - t0
        rate = inst_idx / elapsed if elapsed > 0 else 0
        eta = (total - inst_idx) / rate if rate > 0 else 0
        score_summary = ", ".join(f"{m[:4]}={scores[m]:.3f}" for m in METRICS)
        print(f"  [{model}] {inst_idx}/{total}: {score_summary}  ({elapsed:.0f}s elapsed, ETA {eta:.0f}s)")

    elapsed = time.perf_counter() - t0
    safe = model.replace(":", "_").replace(".", "_")
    out_path = output_dir / f"scores_{safe}.json"
    out_path.write_text(json.dumps(records, indent=2))
    print(f"  Saved {len(records)} records to {out_path}  ({elapsed:.1f}s total)")
    return records

In [ ]:
timing = {}

for model in settings.models:
    safe = model.replace(":", "_").replace(".", "_")
    out_path = eval_dir / f"scores_{safe}.json"

    if not FORCE_RESCORE and out_path.exists():
        existing = json.loads(out_path.read_text())
        nans = _nan_count(existing)
        if existing and nans == 0:
            print(f"SKIP {model}: {len(existing)} valid records in {out_path.name}")
            continue
        reason = f"{nans} NaN scores" if nans > 0 else "empty file"
        print(f"RE-SCORE {model}: {reason} in {out_path.name}, deleting and re-running")
        out_path.unlink()

    if model not in available_models:
        print(f"SKIP {model}: not available in Ollama (pull with: ollama pull {model})")
        continue

    # Restart Ollama for a clean memory slate — avoids state corruption from
    # the previous model and eliminates residual KV-cache allocations.
    restart_ollama(settings.ollama_url)

    # Warm up: load the model and confirm it responds before the scoring loop.
    judge_check = OllamaJudge(model=model, ollama_url=settings.ollama_url, num_ctx=settings.num_ctx)
    if not judge_check.warm_up(timeout=120.0):
        print(f"  SKIP {model}: model failed to load after Ollama restart")
        continue

    print(f"\nScoring with {model} ({len(all_instances)} instances \u00d7 {len(METRICS)} metrics)...")
    t0 = time.perf_counter()
    score_model(model, all_instances, eval_dir)
    timing[model] = time.perf_counter() - t0

if timing:
    print("\n=== Timing summary ===")
    for m, t in timing.items():
        print(f"  {m}: {t/60:.1f} min")
else:
    print("\nAll models skipped (already scored or unavailable).")


## 4. Score summary

In [ ]:
from IPython.display import display

score_files = list(eval_dir.glob("scores_*.json"))
if not score_files:
    print("No score files found. Run the scoring cell above first.")
else:
    all_records = []
    for f in sorted(score_files):
        all_records.extend(json.loads(f.read_text()))

    if not all_records:
        print("Score files found but all are empty — run the scoring cell above.")
    else:
        scores_df = pd.DataFrame(all_records)
        print(f"Total score records: {len(scores_df)}")
        print(f"Models scored: {sorted(scores_df['model'].unique())}")
        print("\nMean score per (model, metric):")
        display(scores_df.groupby(["model", "metric"])["score"].mean().unstack().round(3))

        summary = scores_df.groupby(["model", "metric"])["score"].agg(["mean", "std", "count"]).round(4)
        summary_path = RESULTS_DIR / "02_score_summary.json"
        summary.reset_index().to_json(summary_path, orient="records", indent=2)
        print(f"Saved score summary to {summary_path}")

## 5. Load and inspect human scores

In [ ]:
human_path = ROOT / "data" / "human" / "human_scores.csv"
if human_path.exists():
    human_df = pd.read_csv(human_path)
    print(f"Human scores: {len(human_df)} instances")
    print("\nMean human scores by metric:")
    print(human_df[["context_relevance", "groundedness", "answer_relevance"]].mean().round(3).to_string())
else:
    print("Human scores not found — run notebook 01 first.")

---
**Next:** Run `03_inter_judge_agreement.ipynb` to compute kappa and correlation metrics across all annotators.